In [0]:
from pyspark.sql import functions as F

raw_file_path = "/Volumes/email_triage/source_data/raw/customer_support_tickets.csv"

df_raw = (
    spark.read.format("csv").option("header", "true").option("multiline", "true").option("escape", '"').option("inferSchema", "false").load(raw_file_path)
)

print(f"df_raw has {df_raw.count()}")
display(df_raw.limit(5))
      


In [0]:
df_standardized = df_raw
for col_name in df_raw.columns:
    clean_col_name = col_name.strip().replace(" ", "_").lower()
    df_standardized = df_standardized.withColumnRenamed(col_name, clean_col_name)
    
df_bronze = (
    df_standardized.withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path"))
)

display(df_bronze.limit(5))

In [0]:
target_table = "email_triage.bronze.brz_tickets"

df_bronze.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(target_table)

In [0]:
%sql
SELECT 
    COUNT(*) AS total_records,
    COUNT(DISTINCT ticket_id) AS unique_tickets,
    MIN(`_ingested_at`) AS first_ingested
FROM email_triage.bronze.brz_tickets;